In [7]:
# Load environment variables (API keys)
from dotenv import load_dotenv; load_dotenv()
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
import time, random, requests
from pandas.tseries.offsets import MonthEnd
from pytrends.request import TrendReq
from fredapi import Fred

In [8]:
INTERIM_PATH = Path("../data/interim")
PROCESSED_PATH = Path("../data/processed")

df = pd.read_parquet(INTERIM_PATH / 'returns_monthly_stocks.parquet',
                       engine='pyarrow')
# Ensure that the date is in date time format
df['date'] = pd.to_datetime(df['date'])
# Ensure values are sorted
df = df.sort_values(['ticker','date'])
# Group by ticker
grouped_df = df.groupby('ticker', group_keys=False)


# ======= Momentum =========


# Create Momentum for each of the intervals
df['mom_log_3'] = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(3,min_periods=3).sum())
df['mom_log_6'] = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(6,min_periods=6).sum())
df['mom_log_12'] = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(12,min_periods=12).sum())

# ========== Volatility ===========

# Calculate the std if the lstock for each ticker over 6 and 12 months
df['vol_6'] = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(6,min_periods=6).std())
df['vol_12'] = grouped_df['l_stock'].apply(lambda s: s.shift(1).rolling(12,min_periods=12).std())

# =========== Beta to find slope =============

def rolling_beta(stock, bench, window=36):
    s = stock.shift(1)
    b = bench.shift(1)
    cov_sb = s.rolling(window, min_periods=24).cov(b)
    var_b  = b.rolling(window, min_periods=24).var()
    return cov_sb / var_b

df['beta_36'] = grouped_df.apply(lambda d: rolling_beta(d['l_stock'], d['l_bench']),include_groups=False)

# ============== Reversal and Liquidity ====================

# 1 month reversal
df['rev_1m'] = grouped_df['l_stock'].shift(1)

# Liquidity
df['liquidity'] = grouped_df['volume'].apply(lambda s: np.log(s.clip(lower=1)).shift(1).rolling(12,min_periods=6).mean())

# ============ Save to Parquet =====================
core_path = INTERIM_PATH / "features_stocks_core.parquet"
df.to_parquet(core_path, index=False)
print(f"Saved -> {core_path}")
print(f"Rows: {len(df):,} | Cols: {df.shape[1]}")



# =============== Yearly winsorisation (clips 1% / 99% per calendar year) ========================
def winsorise_yearly(df, exclude_cols=None, p_low=0.01, p_high=0.99):
    """
    Clips continuous numeric columns within each calendar year to [p_low, p_high] quantiles.
    Skips: id/date, labels, and binary dummies (0/1).
    Returns a new DataFrame.
    """
    import pandas as pd
    out = df.copy()
    out["__year__"] = pd.to_datetime(out["date"]).dt.year

    # Exclusions
    exclude_cols = set(exclude_cols or [])
    exclude_cols.update({"date", "ticker", "__year__"})

    # Candidate numeric, non-binary columns
    cand = []
    for c in out.columns:
        if c in exclude_cols:
            continue
        if not pd.api.types.is_numeric_dtype(out[c]):
            continue
        vals = pd.Series(out[c].dropna().unique())
        if len(vals) <= 2 and set(vals.astype(float)).issubset({0.0, 1.0}):
            continue
        cand.append(c)
    if not cand:
        return out.drop(columns="__year__")

    # Compute per-year quantiles via transform (no apply, no warning)
    q_low  = out.groupby("__year__")[cand].transform(lambda s: s.quantile(p_low))
    q_high = out.groupby("__year__")[cand].transform(lambda s: s.quantile(p_high))

    # Clip using row-aligned bounds
    out[cand] = out[cand].clip(q_low, q_high)

    return out.drop(columns="__year__")

Saved -> ../data/interim/features_stocks_core.parquet
Rows: 2,480 | Cols: 16


# Additional Classical features

In [ ]:
# =========== Sharpe-like ratio on excess returns ============
g_ex = df.groupby('ticker')['ex_log']
mu_ex12 = g_ex.transform(lambda s: s.shift(1).rolling(12, min_periods=12).mean())
sd_ex12 = g_ex.transform(lambda s: s.shift(1).rolling(12, min_periods=12).std())

# Where/ if there is 0 make it NaN
df['sharpe_12'] = np.where(sd_ex12 > 0, mu_ex12 / sd_ex12, np.nan)

# =========== 12 month high ======================
px_1 = df.groupby('ticker')['close'].shift(1)
roll_max12 = df.groupby('ticker')['close'].transform(lambda s: s.shift(1).rolling(12, min_periods=12).max())
sma12      = df.groupby('ticker')['close'].transform(lambda s: s.shift(1).rolling(12, min_periods=12).mean())

df['near_high_12'] = px_1 / roll_max12
df['dist_sma_12']  = px_1 / sma12 - 1

# ============== Drawdown Depth =======================
df['drawdown_12'] = px_1 / roll_max12 - 1

# ================= Trackin error ==========================
df['tracking_error_12'] = (
    df.groupby('ticker')['ex_log']
      .transform(lambda s: s.shift(1).rolling(12, min_periods=12,).std())
)

# =============== Up/down capture 36m ====================
def up_down_capture_group(g):
    s = g['l_stock'].shift(1)   # stock log returns known at t-1
    b = g['l_bench'].shift(1)   # bench log returns known at t-1
    up_cap   = s.where(b > 0).rolling(36, min_periods=24).mean()
    down_cap = s.where(b < 0).rolling(36, min_periods=24).mean()
    return pd.DataFrame({'up_cap_36': up_cap, 'down_cap_36': down_cap})

cap = (
    df.groupby('ticker', group_keys=False, as_index=False)[['l_stock','l_bench']]
      .apply(lambda g: up_down_capture_group(g))
)
df[['up_cap_36','down_cap_36']] = cap[['up_cap_36','down_cap_36']]
df['up_down_cap_ratio_36'] = np.where(df['down_cap_36'].abs() > 1e-8,
                                      df['up_cap_36'] / df['down_cap_36'],
                                      np.nan)

# ================== Strutural ========================

# Country flag
df['country_UK'] = df['ticker'].str.endswith('.L').astype('int8')

# Sector one-hots for your 10 names
sector_map = {
    'AAPL':'Tech', 'JPM':'Financials', 'XOM':'Energy', 'PFE':'Healthcare', 'WMT':'Staples',
    'HSBA.L':'Financials', 'BP.L':'Energy', 'TSCO.L':'Staples', 'AZN.L':'Healthcare', 'RR.L':'Industrials'
}
df['sector'] = df['ticker'].map(sector_map)

sector_dummies = pd.get_dummies(df['sector'], prefix='sec', dtype='int8')
df = pd.concat([df, sector_dummies], axis=1)

# Sector-relative 12m momentum
if 'mom_log_12' in df.columns:
    df['sector_mom12_mean'] = df.groupby(['sector','date'])['mom_log_12'].transform('mean')
    df['sector_rel_mom12']  = df['mom_log_12'] - df['sector_mom12_mean']

# ======================== Clean up ====================================

for c in ['gt_ma3','gt_z12','gt_spike']:
    if c not in df.columns:
        df[c] = np.nan

id_cols   = ['date','ticker']
base_cols = ['l_stock','l_bench','ex_log','close','volume']  # handy later
feat_cols = [
    # price/behavior
    'mom_log_3','mom_log_6','mom_log_12','rev_1m','vol_6','vol_12','sharpe_12',
    'near_high_12','dist_sma_12','drawdown_12',
    'beta_36','tracking_error_12','up_cap_36','down_cap_36','up_down_cap_ratio_36',
    'liquidity_12m',
    # structural
    'country_UK','sec_Energy','sec_Financials','sec_Healthcare','sec_Industrials','sec_Staples','sec_Tech',
    # macro (will only keep if merged already)
    'vix_lvl','dxy_lvl','us10y_lvl','brent_d3m',
    # trends
    'gt_ma3','gt_z12','gt_spike',
    # (optional) sector-relative
    'sector_rel_mom12'
]
keep = id_cols + base_cols + [c for c in feat_cols if c in df.columns]
df = df[keep].sort_values(['ticker','date']).copy()

# ==================== SAVE ============================

print("Top-10 NA rates:\n", df.isna().mean().sort_values(ascending=False).head(10))
print(df[['mom_log_12','vol_12','beta_36','near_high_12']].describe())

assert not df.duplicated(['date','ticker']).any(), "Duplicate (date,ticker) rows."

core_path = INTERIM_PATH / "features_stocks_core.parquet"
df.to_parquet(core_path, index=False)
print("Saved (core):", core_path, "| rows:", len(df), "| cols:", df.shape[1])


Top-10 NA rates:
 gt_spike                1.000000
gt_z12                  1.000000
gt_ma3                  1.000000
up_down_cap_ratio_36    1.000000
down_cap_36             1.000000
up_cap_36               0.826613
beta_36                 0.096774
drawdown_12             0.048387
sharpe_12               0.048387
tracking_error_12       0.048387
dtype: float64
        mom_log_12       vol_12      beta_36  near_high_12
count  2360.000000  2360.000000  2240.000000   2360.000000
mean      0.076380     0.066266     0.960874      0.907969
std       0.263618     0.030644     0.467771      0.109860
min      -1.807470     0.022277    -0.176099      0.182687
25%      -0.070619     0.047165     0.654803      0.863700
50%       0.078813     0.059225     0.936437      0.941195
75%       0.226071     0.076934     1.219922      0.998891
max       1.168034     0.334269     3.356211      1.000000
Saved (core): ../data/interim/features_stocks_core.parquet | rows: 2480 | cols: 33


# Fred Api Features 

In [ ]:
# ======== MACRO (FRED) → MERGE → SAVE BASELINE ===============



# --- Guards ---
assert 'df' in globals(), "Build or load your base (date,ticker) dataframe 'df' first."
assert 'INTERIM_PATH' in globals(), "Define INTERIM_PATH (e.g., Path('../data/interim')) before running."

# Pull daily macro series from FRED ---
fred = Fred(api_key=os.getenv("FRED_API_KEY"))

START, END = "2003-01-01", "2025-12-31"             # start early so 2004/2005 windows exist
vix   = fred.get_series('VIXCLS',       observation_start=START, observation_end=END)   # VIX level
brent = fred.get_series('DCOILBRENTEU', observation_start=START, observation_end=END)   # Brent ($/bbl)
dxy   = fred.get_series('DTWEXBGS',     observation_start=START, observation_end=END)   # Broad USD
us10y = fred.get_series('DGS10',        observation_start=START, observation_end=END)   # 10Y yield (%)
dexuk  = fred.get_series('DEXUSUK',      observation_start=START, observation_end=END)   # USD per 1 GBP


# Forward-fill and month-end resample
daily = (
    pd.concat(
        [vix.rename('vix_lvl'),
         brent.rename('brent'),
         dxy.rename('dxy_lvl'),
         us10y.rename('us10y_lvl'),
         dexuk.rename('dex_usuk'),
        ], axis=1
    )
    .sort_index()
    .ffill()
)
daily.index = pd.to_datetime(daily.index)

# Month-end resample (new alias) and align to exact month-end
macro_m = daily.resample('ME').last()
macro_m.index = macro_m.index + MonthEnd(0)

# Engineer features (lag 1 month to avoid look-ahead)
macro_m['brent_d3m'] = np.log(macro_m['brent'].clip(lower=1e-6)).diff(3)

# Lag everything by 1 month so values at t are known at t-1
for c in ['vix_lvl','dxy_lvl','us10y_lvl', 'dex_usuk', 'brent_d3m']:
    macro_m[c] = macro_m[c].shift(1)

# Keep and reset index
macro_m.index.name = 'date'
macro_m = macro_m[['vix_lvl','dxy_lvl','us10y_lvl', 'dex_usuk','brent_d3m']].reset_index()
macro_m['date'] = pd.to_datetime(macro_m['date'])

# QA (macro table)
assert not macro_m.duplicated('date').any(), "Duplicate month in macro_m."
print("Macro tail:\n", macro_m.tail(3))

# Save reusable macro parquet ---
macro_path = INTERIM_PATH / "macro_monthly.parquet"
macro_m.to_parquet(macro_path, index=False)
print("Saved macro:", macro_path)

# Merge macro onto your features df
df = df.copy()
df['date'] = pd.to_datetime(df['date']) + MonthEnd(0)   # ensure month-end alignment

df = df.merge(macro_m, on='date', how='left', validate='many_to_one')
print("Merged macro. Rows:", len(df))

# Define keep schema
if 'id_cols' not in globals():
    id_cols   = ['date','ticker']
if 'base_cols' not in globals():
    base_cols = ['l_stock','l_bench','ex_log','close','volume']
if 'feat_cols' not in globals():
    feat_cols = [
        # price / behaviour
        'mom_log_3','mom_log_6','mom_log_12','rev_1m','vol_6','vol_12','sharpe_12',
        'near_high_12','dist_sma_12','drawdown_12',
        'beta_36','tracking_error_12','up_cap_36','down_cap_36','up_down_cap_ratio_36',
        'liquidity_12m',
        # structural
        'country_UK','sec_Energy','sec_Financials','sec_Healthcare','sec_Industrials','sec_Staples','sec_Tech',
        # macro
        'vix_lvl','dxy_lvl','us10y_lvl','dex_usuk','brent_d3m',
        # trends
        'gt_ma3','gt_z12','gt_spike',
        # sector-relative
        'sector_rel_mom12'
    ]

# Ensure Trends placeholders exist so keep-list never KeyErrors
for c in ['gt_ma3','gt_z12','gt_spike']:
    if c not in df.columns:
        df[c] = np.nan

# Trim → QA → Save BASELINE
df = df.sort_values(['ticker','date'])
keep = id_cols + base_cols + [c for c in feat_cols if c in df.columns]
df = df[keep].copy()

# Uniqueness & simple QA
assert not df.duplicated(['date','ticker']).any(), "Duplicate (date,ticker) rows."
print("Top-10 NA rates:\n", df.isna().mean().sort_values(ascending=False).head(10))
print(df[['mom_log_12','vol_12','beta_36','near_high_12','vix_lvl']].describe())

# Save BASELINE features (price/behaviour + macro + structure; no Trends)
out_path = INTERIM_PATH / "features_stocks.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved -> {out_path}")
print(f"Rows: {len(df):,} | Cols: {df.shape[1]}")


Macro tail:
           date  vix_lvl   dxy_lvl  us10y_lvl  dex_usuk  brent_d3m
271 2025-08-31    16.72  122.1088       4.37    1.3220   0.147342
272 2025-09-30    15.36  120.6028       4.23    1.3511   0.053134
273 2025-10-31    16.28  120.5624       4.16    1.3443   0.005415
Saved macro: ../data/interim/macro_monthly.parquet
Merged macro. Rows: 2480
Top-10 NA rates:
 gt_spike                1.000000
gt_z12                  1.000000
gt_ma3                  1.000000
up_down_cap_ratio_36    1.000000
down_cap_36             1.000000
up_cap_36               0.826613
beta_36                 0.096774
tracking_error_12       0.048387
sharpe_12               0.048387
dxy_lvl                 0.048387
dtype: float64
        mom_log_12       vol_12      beta_36  near_high_12      vix_lvl
count  2360.000000  2360.000000  2240.000000   2360.000000  2480.000000
mean      0.076380     0.066266     0.960874      0.907969    19.285121
std       0.263618     0.030644     0.467771      0.109860     8.130

# GOOGLE TRENDS

In [ ]:

# ---------- Paths ----------
try:
    INTERIM_PATH
except NameError:
    INTERIM_PATH = Path("../data/interim")

TRENDS_DIR = INTERIM_PATH / "external_trends"   # folder with your 2 CSVs
files = sorted(TRENDS_DIR.glob("*.csv"))
assert files, f"No CSV files found in {TRENDS_DIR}. Put your downloads there."

# ---------- Normalisation + mapping ----------
def normalise_query(col: str) -> str:
    """
    Normalise Google Trends header to be mapping-friendly:
    - lowercases
    - strips parentheses
    - removes region/category tails after colon or dash (': UK', ' - United States', etc.)
    - collapses whitespace
    """
    if not isinstance(col, str):
        return col
    s = col.strip().lower()
    s = re.sub(r"\(.*?\)", "", s)                      # remove (...) blocks
    s = re.split(r"\s*[:\-–—]\s*", s)[0]               # cut off after colon/dash
    s = re.sub(r"\s+", " ", s).strip()                 # collapse spaces
    return s

PATTERNS = {
    'AAPL'  : [r'\bapple\b'],
    'JPM'   : [r'\bjp\s*morgan\b', r'\bjpm\b'],
    'XOM'   : [r'\bexxon\b'],
    'PFE'   : [r'\bpfizer\b'],
    'WMT'   : [r'\bwalmart\b'],
    'HSBA.L': [r'\bhsbc\b'],
    'BP.L'  : [r'\bbp\b'],
    'TSCO.L': [r'\btesco\b'],
    'AZN.L' : [r'\bastrazeneca\b', r'\bazn\b'],
    'RR.L'  : [r'\brolls[-\s]?royce\b', r'\brr\b', r'\brolls\b'],
}

def map_query_to_ticker(q: str) -> str | None:
    if not isinstance(q, str):
        return None
    qn = normalise_query(q)
    for tkr, pats in PATTERNS.items():
        for pat in pats:
            if re.search(pat, qn):
                return tkr
    return None

# ---------- Load CSVs → long format ----------
all_queries = set()
frames = []

for f in files:
    # Find the table start row ("Month" or "Week")
    with open(f, "r", encoding="utf-8") as fh:
        lines = fh.readlines()
    start = next(i for i, L in enumerate(lines) if L.strip().startswith(("Month", "Week")))
    raw = pd.read_csv(f, skiprows=start)

    # First column is the time axis
    time_col = raw.columns[0]
    raw = raw.rename(columns={time_col: "date"})

    # Normalise query headers (handles regional/category suffixes)
    ren = {c: normalise_query(c) for c in raw.columns if c != "date"}
    raw = raw.rename(columns=ren)
    all_queries.update([c for c in raw.columns if c != "date"])

    # Parse date & snap to month-end for alignment
    raw["date"] = pd.to_datetime(raw["date"], errors="coerce") + MonthEnd(0)
    raw = raw.dropna(subset=["date"])

    # Long format: one row per (date, query)
    long = raw.melt(id_vars="date", var_name="query", value_name="gt_raw")

    # Clean "<1" → 0, cast to float
    long["gt_raw"] = pd.to_numeric(long["gt_raw"].replace("<1", 0), errors="coerce")

    # Map query → ticker via regex patterns
    long["ticker"] = long["query"].apply(map_query_to_ticker)
    long = long.dropna(subset=["ticker"])

    frames.append(long[["date", "ticker", "gt_raw"]])

# Combine both CSVs and dedupe by (date,ticker)
gt = (pd.concat(frames, ignore_index=True)
        .groupby(["ticker", "date"], as_index=False)["gt_raw"].mean()
        .sort_values(["ticker", "date"]))

print("Distinct normalised query headers found in CSVs:")
print(sorted(all_queries))

# ---------- Feature engineering (per ticker) ----------
g = gt.groupby("ticker")["gt_raw"]
gt["gt_ma3"]    = g.transform(lambda s: s.rolling(3,  min_periods=3).mean())
gt["gt_mean12"] = g.transform(lambda s: s.rolling(12, min_periods=12).mean())
gt["gt_std12"]  = g.transform(lambda s: s.rolling(12, min_periods=12).std(ddof=0))
gt["gt_z12"]    = (gt["gt_raw"] - gt["gt_mean12"]) / gt["gt_std12"].replace(0, np.nan)
gt["gt_spike"]  = (gt["gt_z12"] > 2).astype("int8")

# Critical: LAG by 1 month so features at t use info up to t-1
for c in ["gt_ma3", "gt_z12", "gt_spike"]:
    gt[c] = gt.groupby("ticker")[c].shift(1)

gt_feat = gt[["date", "ticker", "gt_ma3", "gt_z12", "gt_spike"]].drop_duplicates()
assert not gt_feat.duplicated(["date", "ticker"]).any(), "Duplicate (date,ticker) in Trends features."

print("\nCounts by ticker in Trends features (post-lag):")
print(gt_feat["ticker"].value_counts(dropna=False).sort_index())

# ---------- Load base df (features_stocks) ----------
if "df" not in globals():
    base_path = INTERIM_PATH / "features_stocks.parquet"
    assert base_path.exists(), f"Missing base features file: {base_path}"
    df = pd.read_parquet(base_path)

# Defensive: ensure month-end dates
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)

# ---------- Safe merge (avoids KeyErrors and _x/_y suffixes) ----------
TR_COLS = ["gt_ma3", "gt_z12", "gt_spike"]

# If placeholders already exist, drop them to avoid suffix clashes
to_drop = [c for c in TR_COLS if c in df.columns]
if to_drop:
    df = df.drop(columns=to_drop)

before = len(df)
df = df.merge(gt_feat, on=["date", "ticker"], how="left", validate="many_to_one", suffixes=("", "_tr"))
print(f"\nMerged manual Trends → rows: {before} → {len(df)}")

# Coalesce any accidental suffixes (defensive)
for base in TR_COLS:
    x, y = f"{base}_x", f"{base}_y"
    if x in df.columns or y in df.columns:
        df[base] = df.get(y).combine_first(df.get(x))
        df.drop(columns=[col for col in (x, y) if col in df.columns], inplace=True)

# Ensure columns exist even if nothing merged (pipeline stays stable)
for c in TR_COLS:
    if c not in df.columns:
        df[c] = np.nan

print("Non-null counts:", {c: int(df[c].notna().sum()) for c in TR_COLS})

# If still nothing merged, print mismatch sets
if df[TR_COLS].notna().sum().sum() == 0:
    base_set = set(df['ticker'].unique())
    feat_set = set(gt_feat['ticker'].unique())
    print("\n[Diag] No Trends merged.")
    print("  In base only:", sorted(base_set - feat_set))
    print("  In trends only:", sorted(feat_set - base_set))

# --- Winsorise v2 (no pageviews) before saving ---
exclude_for_v2 = {
    # ids / labels / raw references
    "date","ticker","l_stock","l_bench","ex_log","close","volume",
    # keep country + sector dummies untouched (binary detection will skip them anyway)
    "country_UK","sec_Energy","sec_Financials","sec_Healthcare","sec_Industrials","sec_Staples","sec_Tech"
}

df = winsorise_yearly(df, exclude_cols=exclude_for_v2)

# (QA)
print("[v2] Winsorised describe (sanity):")
print(df[["mom_log_12","vol_12","beta_36","near_high_12"]].describe())


# ---------- Save v2 ----------
out_v2 = INTERIM_PATH / "features_stocks_v2.parquet"
df.to_parquet(out_v2, index=False)
print("\nSaved:", out_v2, "| rows:", len(df), "| cols:", df.shape[1])

# ---------- Quick QA ----------
try:
    last_tr = (df[df["gt_ma3"].notna()]
               .sort_values("date")
               .groupby("ticker").tail(1)[["ticker", "date", "gt_z12", "gt_spike"]])
    print("\nLast month with Trends per ticker:")
    print(last_tr.to_string(index=False))
except Exception as e:
    print("\n(Info) Could not print last-month Trends per ticker:", e)

Distinct normalised query headers found in CSVs:
['apple stock', 'astrazeneca share price', 'bp share price', 'exxon mobil stock', 'hsbc share price', 'jpmorgan stock', 'pfizer stock', 'rolls', 'tesco share price', 'walmart stock']

Counts by ticker in Trends features (post-lag):
ticker
AAPL      262
AZN.L     262
BP.L      262
HSBA.L    262
JPM       262
PFE       262
RR.L      262
TSCO.L    262
WMT       262
XOM       262
Name: count, dtype: int64

Merged manual Trends → rows: 2480 → 2480
Non-null counts: {'gt_ma3': 2480, 'gt_z12': 1845, 'gt_spike': 2480}
[v2] Winsorised describe (sanity):
        mom_log_12       vol_12      beta_36  near_high_12
count  2360.000000  2360.000000  2240.000000   2360.000000
mean      0.076389     0.066229     0.960672      0.908340
std       0.258097     0.030350     0.465567      0.108673
min      -1.297632     0.024904    -0.161932      0.292103
25%      -0.070619     0.047165     0.654803      0.863700
50%       0.078813     0.059225     0.936437   

# Wikipedia Pageviews

In [12]:
# Load your latest feature file (with/without Trends)
base_candidates = [
    INTERIM_PATH / "features_stocks_v2.parquet",
    INTERIM_PATH / "features_stocks.parquet"
]
base_path = next((p for p in base_candidates if p.exists()), None)
assert base_path is not None, "No base features file found (run earlier cells first)."
df = pd.read_parquet(base_path)
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)

# Ticker → canonical enwiki article titles (case & underscores matter)
WIKI_TITLES = {
    "AAPL":   "Apple_Inc.",
    "JPM":    "JPMorgan_Chase",
    "XOM":    "ExxonMobil",
    "PFE":    "Pfizer",
    "WMT":    "Walmart",
    "HSBA.L": "HSBC",
    "BP.L":   "BP",
    "TSCO.L": "Tesco",
    "AZN.L":  "AstraZeneca",
    "RR.L":   "Rolls-Royce_Holdings",
}

# Pageviews exist from ~2015 forward
START, END = "20150701", "20251231"

# REQUIRED by Wikimedia: a descriptive UA with contact
HEADERS = {
    "User-Agent": "UniDissertationStocks/1.0 (1234.email@university.ac.uk)",
    "Accept": "application/json"
}

def fetch_pageviews_monthly(title: str,
                            start=START, end=END,
                            project="en.wikipedia",
                            access="all-access",
                            agent="user",
                            granularity="monthly",
                            max_retries=4, base_pause=0.6) -> pd.DataFrame:
    """
    Returns ['date','views'] monthly (month-end aligned). Retries on 429/5xx.
    """
    url = (
        "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        f"{project}/{access}/{agent}/{title}/{granularity}/{start}/{end}"
    )

    pause = base_pause
    last_err = None
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            # Handle rate-limit / transient errors with backoff
            if r.status_code in (429, 503, 502, 500):
                raise requests.HTTPError(f"{r.status_code} {r.reason}")
            r.raise_for_status()
            data = r.json().get("items", [])
            if not data:
                return pd.DataFrame(columns=["date","views"])
            out = pd.DataFrame({
                "date": pd.to_datetime([str(it["timestamp"])[:8] for it in data]),
                "views": [it.get("views", np.nan) for it in data]
            })
            out["date"] = out["date"] + MonthEnd(0)
            return out[["date","views"]]
        except Exception as e:
            last_err = e
            if attempt < max_retries - 1:
                # exponential backoff with jitter
                sleep_s = pause * (2 ** attempt) + np.random.uniform(0.0, 0.4)
                print(f"[{title}] Retry {attempt+1}/{max_retries-1} after error: {e}. Sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)
            else:
                print(f"[{title}] FAILED after retries: {e}")
                return pd.DataFrame(columns=["date","views"])

frames = []
for tkr, title in WIKI_TITLES.items():
    pv = fetch_pageviews_monthly(title)
    if pv.empty:
        print(f"[WARN] No pageviews for {tkr} ({title})")
        continue
    pv["ticker"] = tkr
    frames.append(pv[["date","ticker","views"]])
    # be polite between different pages
    time.sleep(0.4)

assert frames, "No pageviews pulled for any ticker."
pv_all = (pd.concat(frames, ignore_index=True)
            .sort_values(["ticker","date"])
            .groupby(["ticker","date"], as_index=False)["views"].sum())

# --- Feature engineering (per ticker) ---
g = pv_all.groupby("ticker")["views"]
pv_all["pv_ma3"]    = g.transform(lambda s: s.rolling(3,  min_periods=3).mean())
pv_all["pv_mean12"] = g.transform(lambda s: s.rolling(12, min_periods=12).mean())
pv_all["pv_std12"]  = g.transform(lambda s: s.rolling(12, min_periods=12).std(ddof=0))
pv_all["pv_z12"]    = (pv_all["views"] - pv_all["pv_mean12"]) / pv_all["pv_std12"].replace(0, np.nan)
pv_all["pv_spike"]  = (pv_all["pv_z12"] > 2).astype("int8")

# LAG by 1 month (avoid leakage)
for c in ["pv_ma3","pv_z12","pv_spike"]:
    pv_all[c] = pv_all.groupby("ticker")[c].shift(1)

pv_feat = pv_all[["date","ticker","pv_ma3","pv_z12","pv_spike"]].drop_duplicates()
assert not pv_feat.duplicated(["date","ticker"]).any(), "Duplicate (date,ticker) in pv_feat."

# Merge onto your features
before = len(df)
df = df.merge(pv_feat, on=["date","ticker"], how="left", validate="many_to_one")
print(f"Merged Wikipedia Pageviews → rows: {before} → {len(df)}")
print("Non-null (pv) counts:", {c:int(df[c].notna().sum()) for c in ["pv_ma3","pv_z12","pv_spike"]})

# Quick QA
last_pv = (df[df["pv_ma3"].notna()]
           .sort_values("date")
           .groupby("ticker").tail(1)[["ticker","date","pv_z12","pv_spike"]])
print("\nLast month with PV features per ticker:")
print(last_pv.to_string(index=False))

# --- Winsorise v3 (with pageviews) before saving ---
exclude_for_v3 = {
    "date","ticker","l_stock","l_bench","ex_log","close","volume",
    "country_UK","sec_Energy","sec_Financials","sec_Healthcare","sec_Industrials","sec_Staples","sec_Tech"
    # Note: binary pv flags (e.g., pv_spike) will be auto-skipped by the helper
}

df = winsorise_yearly(df, exclude_cols=exclude_for_v3)

# (QA)
print("[v3] Winsorised describe (sanity):")
keys = [c for c in ["mom_log_12","vol_12","beta_36","near_high_12","pv_ma3","pv_z12"] if c in df.columns]
print(df[keys].describe())


# Save v3
out_v3 = INTERIM_PATH / "features_stocks_v3.parquet"
df.to_parquet(out_v3, index=False)
print("Saved:", out_v3, "| rows:", len(df), "| cols:", df.shape[1])


Merged Wikipedia Pageviews → rows: 2480 → 2480
Non-null (pv) counts: {'pv_ma3': 1200, 'pv_z12': 1110, 'pv_spike': 1220}

Last month with PV features per ticker:
ticker       date    pv_z12  pv_spike
   JPM 2025-09-30 -0.062405       0.0
TSCO.L 2025-09-30 -0.219126       0.0
  RR.L 2025-09-30 -1.258765       0.0
   PFE 2025-09-30 -0.755133       0.0
HSBA.L 2025-09-30  0.117597       0.0
  BP.L 2025-09-30 -0.172202       0.0
 AZN.L 2025-09-30 -0.453406       0.0
  AAPL 2025-09-30 -1.076519       0.0
   WMT 2025-09-30 -0.681599       0.0
   XOM 2025-09-30  0.017559       0.0
[v3] Winsorised describe (sanity):
        mom_log_12       vol_12      beta_36  near_high_12         pv_ma3  \
count  2360.000000  2360.000000  2240.000000   2360.000000    1200.000000   
mean      0.076395     0.066217     0.960647      0.908412  106027.416553   
std       0.257667     0.030277     0.465343      0.108474  104023.116712   
min      -1.284227     0.024928    -0.153151      0.292139   16073.216667   
2